# 15 — Calibration + logit-space blend re-tuning (LOFO-validated)

**What notebook 14 found (2026-09-10)**: the honest 5-way CNN ensemble
already beats the old single-checkpoint gate badly (log loss 0.4127 vs.
0.4520 -- a bigger jump than any single rung-4 experiment). The existing
blend (`w_cnn=0.70`, still probability-space, still the single seed-42
baseline OOF) barely improves on that (0.4119) despite a real AUROC gain
(0.8915 -> 0.9081) -- and its ECE roughly doubles (0.0316 -> 0.0747). That's
the signature of two differently-scaled probability sources (a from-scratch
sigmoid CNN and a from-scratch sklearn logistic regression) being blended
in probability space with no recalibration: ranking improves, calibration
gets worse, and in a calibration-sensitive metric like log loss the two
roughly cancel out.

Compared against the real leaderboard (log loss 0.4648, AUROC 0.8796): AUROC
dropped 0.0285 on the real holdout (not negligible -- some of the gap is
real generalization loss, not pure calibration) but the blend's ECE was
*already* elevated in-distribution (0.0747, computed purely on OOF -- no
real-test-set access needed to see this). So this isn't a clean "it's all
calibration" story, but there is a real, already-visible calibration defect
worth fixing regardless of how much of the leaderboard gap it closes.

**This notebook (roadmap items 2 + 4, done together since they interact --
Opus's own suggestion)**:
1. Recomputes the classical baseline's OOF on all 5 CNN repeat-seeds' own
   fold splits (previously only seed 42's existed -- `baseline_oof_seed_match.npy`
   -- so every other repeat's LOFO comparison was blending against a
   slightly fold-mismatched baseline OOF; still honest, just not as tightly
   paired as it could be).
2. LOFO-validated joint grid search over (temperature_cnn, temperature_baseline,
   blend_weight) in **logit space**: for each held-out repeat, pick the
   triple that minimizes mean log loss on the other 4 repeats, score it on
   the held-out one -- same discipline as notebook 07's own blend-weight
   LOFO cell, extended to 3 parameters instead of 1.
3. Reports the final candidate (mean of the 5 LOFO-selected triples, applied
   to the full 5-way-ensembled CNN + baseline OOF) alongside notebook 14's
   uncalibrated numbers, so the decision is a direct before/after.

**Data handling**: loads real row-level labels and OOF prediction arrays
(recomputing the classical baseline also touches `baseline_features.csv`,
already-extracted aggregate features, not raw pixel data), so per the
AI-assistant data rule (`README.md`) this is **[RUN ME]** — run it
yourself, share back only the printed aggregate numbers. CPU-only, no GPU,
no volume cache -- seconds to low tens of seconds for the grid search.

In [ ]:
# [RUN ME] -- recomputes the classical ComBat baseline's OOF on each of
# the 5 CNN repeat-seeds' own fold split (notebook 07 only ever did this
# for seed=config.SEED=42). CPU-only, seconds. Self-contained, does not
# assume any earlier cell ran in this kernel session.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import config
import evaluate
import model

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
y_true = np.array(labels)

baseline_feat_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")
baseline_feat_df = baseline_feat_df.set_index(config.UID_COLUMN).loc[uids].reset_index()
baseline_X = baseline_feat_df[["abs_asym", "striatal_ratio"]].to_numpy()
baseline_y = baseline_feat_df[config.TARGET_COLUMN].to_numpy()
baseline_family = baseline_feat_df["inplane_family"].to_numpy()
assert np.array_equal(baseline_y, y_true), "baseline_features.csv row order must match labels_df merge"

repeat_seeds = list(range(config.SEED, config.SEED + 5))
baseline_oof_repeats = []
for seed in repeat_seeds:
    cache_path = config.DATA_PROCESSED / f"baseline_oof_seed{seed}.npy"
    if cache_path.exists():
        baseline_oof_repeats.append(np.load(cache_path))
        print(f"seed={seed}: loaded existing {cache_path.name}")
        continue
    baseline_oof = np.zeros(len(uids))
    folds = evaluate.make_folds(baseline_y, baseline_family, n_splits=config.N_FOLDS, random_state=seed)
    for train_idx, test_idx in folds:
        pipeline = model.build_combat_baseline()
        pipeline.fit(baseline_X[train_idx], baseline_y[train_idx], baseline_family[train_idx])
        baseline_oof[test_idx] = pipeline.predict_proba(baseline_X[test_idx], baseline_family[test_idx])[:, 1]
    pooled_ll = evaluate.log_loss_score(baseline_y, baseline_oof)
    print(f"seed={seed}: baseline OOF pooled log loss={pooled_ll:.4f}")
    np.save(cache_path, baseline_oof)
    baseline_oof_repeats.append(baseline_oof)

cnn_oof_repeats = [np.load(config.DATA_PROCESSED / f"rung3_oof_seed{s}.npy") for s in repeat_seeds]
print(f"\n{len(repeat_seeds)} repeats ready: CNN + seed-matched baseline OOF, each on its own fold split.")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# LOFO-validated joint grid search over (T_cnn, T_baseline, w) in logit
# space. A fast manual log loss (for the grid search) plus the official
# evaluate.log_loss_score (for every reported/held-out number) -- same
# split as notebook 07's own weight-only LOFO cell, extended to 3 params.
T_GRID = np.arange(0.5, 2.01, 0.1)
W_GRID = np.arange(0.0, 1.01, 0.05)
EPS = 1e-6


def to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def calibrated_blend(cnn_p, baseline_p, t_cnn, t_baseline, w):
    combined_logit = w * (to_logit(cnn_p) / t_cnn) + (1 - w) * (to_logit(baseline_p) / t_baseline)
    return 1.0 / (1.0 + np.exp(-combined_logit))


def fast_log_loss(y, p):
    p = np.clip(p, EPS, 1 - EPS)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


cnn_logits = [to_logit(oof) for oof in cnn_oof_repeats]
baseline_logits = [to_logit(oof) for oof in baseline_oof_repeats]

honest_scores = []
selected_params = []
for held_out_i in range(len(repeat_seeds)):
    selection = [i for i in range(len(repeat_seeds)) if i != held_out_i]
    best = None
    for t_cnn in T_GRID:
        cnn_scaled = [cnn_logits[i] / t_cnn for i in selection]
        for t_base in T_GRID:
            base_scaled = [baseline_logits[i] / t_base for i in selection]
            for w in W_GRID:
                mean_ll = np.mean([
                    fast_log_loss(y_true, 1.0 / (1.0 + np.exp(-(w * cs + (1 - w) * bs))))
                    for cs, bs in zip(cnn_scaled, base_scaled)
                ])
                if best is None or mean_ll < best[0]:
                    best = (mean_ll, float(t_cnn), float(t_base), float(w))
    _, t_cnn, t_base, w = best
    held_out_probs = calibrated_blend(cnn_oof_repeats[held_out_i], baseline_oof_repeats[held_out_i], t_cnn, t_base, w)
    held_out_score = evaluate.log_loss_score(y_true, held_out_probs)
    honest_scores.append(held_out_score)
    selected_params.append((t_cnn, t_base, w))
    print(f"  held-out repeat {held_out_i} (seed={repeat_seeds[held_out_i]}): "
          f"selected T_cnn={t_cnn:.2f}, T_baseline={t_base:.2f}, w={w:.2f} on the other 4, "
          f"scored {held_out_score:.4f} on this one")

honest_scores = np.array(honest_scores)
print(f"\nLOFO calibrated-blend: mean={honest_scores.mean():.4f}, sd={honest_scores.std(ddof=1):.4f}")
print("for comparison -- notebook 14's uncalibrated numbers: "
      "CNN ensemble=0.4127, blend (w_cnn=0.70, probability space)=0.4119")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Build the actual production CANDIDATE: apply the mean of the 5
# LOFO-selected (T_cnn, T_baseline, w) triples to the full 5-way-ensembled
# CNN + 5-way-ensembled baseline OOF (mirrors notebook 14's ensembling,
# now with calibration + a re-tuned logit-space blend on top).
final_t_cnn = float(np.mean([p[0] for p in selected_params]))
final_t_baseline = float(np.mean([p[1] for p in selected_params]))
final_w = float(np.mean([p[2] for p in selected_params]))
print(f"candidate params (mean of 5 LOFO-selected triples): "
      f"T_cnn={final_t_cnn:.2f}, T_baseline={final_t_baseline:.2f}, w={final_w:.2f}")

cnn_ensemble_oof = np.mean(cnn_oof_repeats, axis=0)
baseline_ensemble_oof = np.mean(baseline_oof_repeats, axis=0)
candidate_oof = calibrated_blend(cnn_ensemble_oof, baseline_ensemble_oof, final_t_cnn, final_t_baseline, final_w)

candidate_scores = evaluate.combined_score(y_true, candidate_oof)
print(f"\ncandidate (calibrated, re-tuned blend): log loss={candidate_scores['log_loss']:.4f}  "
      f"AUROC={candidate_scores['auroc']:.4f}  ECE={candidate_scores['ece']:.4f}")
print("notebook 14 reference -- CNN ensemble: log loss=0.4127 AUROC=0.8915 ECE=0.0316")
print("notebook 14 reference -- blend (uncalibrated, w_cnn=0.70): log loss=0.4119 AUROC=0.9081 ECE=0.0747")
print("real leaderboard (first submission): log loss=0.4648 AUROC=0.8796")

**What we're looking for:** does a properly LOFO-validated, logit-space,
jointly-calibrated blend beat notebook 14's uncalibrated ensemble numbers by
more than noise -- and does its ECE come back down toward the CNN-alone
level (0.0316) instead of the blend's uncalibrated 0.0747?

**What we found:** LOFO held-out scores (single-repeat granularity, same as
notebook 07's original blend-weight LOFO): [0.4100, 0.4053, 0.3852, 0.4043,
0.3984], mean=0.4006, sd=0.0096. **4 of 5 folds independently selected
nearly identical params** (T_cnn=1.00, T_baseline=0.50, w=0.70; fold 3
selected 0.90/0.60/0.65) -- high cross-fold agreement, not a
noise-fit parameter. Compared apples-to-apples against notebook 07's
original uncalibrated single-weight LOFO blend (mean=0.4250, sd=0.0099):
**improvement of -0.0244, ~2.5x the pooled sd** -- a real, well-supported
win by this project's own established bar (same magnitude of evidence as
the w_cnn=0.70 decision itself). The "candidate" cell's pooled number
(0.3842) is directionally consistent but has mild optimism (the averaged
triple partially reflects every repeat's own data through the other folds'
selection) -- the honest LOFO mean (0.4006) above is the number to trust.
T_cnn≈1.0 confirms the CNN was already well-calibrated (matches its low
0.0316 ECE); T_baseline≈0.5 is the real finding -- the classical baseline's
logits were badly scale-mismatched against the CNN's, which is exactly what
was inflating the uncalibrated blend's ECE to 0.0747.

**Decision / next step:** this clears the bar -- logit-space calibration
(T_cnn≈1.0, T_baseline≈0.5, w≈0.70) is adopted as a real improvement over
the plain probability-space blend. Before wiring this into
`submission_src/main.py`/`src/submission.py::combine_predictions`, do
roadmap item 3 first (ensemble across the 125 rung-3+rung-4 checkpoints
already on disk) since it changes the CNN-ensemble's OOF and these exact
calibration params should be re-fit against whatever the final ensemble
composition turns out to be, not locked in prematurely. See
`project_dat_parkinson_strategic_roadmap.md`.